# Notebook 04 — H2 to H4 Moderation Models

**Dissertation:** *Beyond Money: Who Responds to Intergenerational Appeals for Sustainable Cooperation?*

This notebook evaluates Hypotheses 2–4 using a three-level mixed-effects linear probability model. The outcome variable is the round-level indicator of green choice behaviour `extract_green_choice`. Random intercepts are estimated for sessions and participants, reflecting decision rounds nested within participants and participants nested within sessions — the same specification used for H1 in Notebook 03.

- **H2**: the positive effect of intergenerational awareness on green choice is stronger for liberal than for conservative participants, relative to participants who did not receive intergenerational awareness.
- **H3**: as H2, moderated by legacy concern instead of ideology (Prolific only — legacy concern was not collected in MTurk).
- **H4**: as H2, moderated by environmental concern instead of ideology (Prolific only).

Treatment enters as a four-level factor with `NIT&NGI` as the reference category, so each contrast is estimated and interpretable on its own terms. The `T.2` contrast (IT only vs. `NIT&NGI`) is the primary test of each hypothesis, since it isolates intergenerational awareness without the green-information/monetary framing present in `IT&GI`.

Each hypothesis is estimated in two forms, following the logic established for H1: a **primary** specification (lean, confirmatory) and a **robustness check** adding the full pre-treatment demographic covariate set (age, gender, education, ethnicity, trust in scientists, and the Prolific-only risk and reward-preference controls). None of `liberal_z`, `legacy_z`, or `env_concern_z` are randomly assigned, so this check guards against demographic confounding of the moderator terms.

H4 additionally carries a **climate-belief robustness check**. The environmental-concern composite `env_concern_z` is a multi-item value-orientation measure; four further single-item climate measures were collected in the Prolific study — climate worry (`environment_worry`), personal responsibility (`environment_responsibility`), belief that the climate is changing (`environment_changing`), and belief that change is a natural rather than human-driven process (`env_natural_process_ctrl`, the climate-denial response coded missing during preparation). These capture climate cognition distinct from the biospheric-value content of the composite. The climate-controls specification augments the primary H4 model with these four measures to establish that any environmental-concern moderation is not an artefact of correlated climate beliefs. The continuous items are z-standardised for scale comparability; `environment_changing` enters as a factor given its small ordered-category structure.


## 1 · Setup


In [1]:
# Cell 1: Imports and global configuration
import warnings
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.regression.mixed_linear_model import MixedLM

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 100)

os.makedirs('outputs/tables', exist_ok=True)
os.makedirs('outputs/models', exist_ok=True)

TREAT_MAP = {1: 'IT&GI', 2: 'IT', 3: 'GI', 4: 'NIT&NGI'}
TREAT_ORDER = ['IT&GI', 'IT', 'GI', 'NIT&NGI']

print('Environment ready.')


Environment ready.


## 2 · Load Analytic Datasets

The inputs are the analytic CSVs written by Notebook 01. No additional row deletion is introduced here.


In [2]:
# Cell 2: Load analytic datasets
mt = pd.read_csv('data/mturk_clean.csv')
pr = pd.read_csv('data/prolific_clean.csv')

def prepare_common_fields(df):
    df = df.copy()
    df['treatment_str'] = df['treatment'].map(TREAT_MAP)
    df['round_c'] = df['round'] - df['round'].mean()
    return df

mt = prepare_common_fields(mt)
pr = prepare_common_fields(pr)

# Z-standardise the continuous single-item climate measures used in the H4
# climate-belief robustness check (Prolific only). environment_changing is
# left on its original scale and enters as a factor in the formula.
CLIMATE_CONTINUOUS = ['environment_worry', 'environment_responsibility', 'env_natural_process_ctrl']
for col in CLIMATE_CONTINUOUS:
    if col in pr.columns:
        pr[f'{col}_z'] = (pr[col] - pr[col].mean()) / pr[col].std(ddof=1)

print(f"MTurk: {len(mt):,} rows | {mt['participantcode'].nunique():,} participants | {mt['sessioncode'].nunique():,} sessions")
print(f"Prolific: {len(pr):,} rows | {pr['participantcode'].nunique():,} participants | {pr['sessioncode'].nunique():,} sessions")

# env_natural_process_ctrl carries missing values by design (climate-denial
# response set to NaN in preparation); report the resulting analytic-N impact
# for the climate-controls specification, which listwise-drops those rows.
n_missing_npc = pr['env_natural_process_ctrl'].isna().sum() if 'env_natural_process_ctrl' in pr.columns else 0
n_part_npc = pr.loc[pr['env_natural_process_ctrl'].isna(), 'participantcode'].nunique() if n_missing_npc else 0
print(f"env_natural_process_ctrl missing: {n_missing_npc:,} rows across {n_part_npc} participants "
      f"(climate-denial responses; excluded from the H4 climate-controls model only)")


MTurk: 10,220 rows | 1,022 participants | 308 sessions
Prolific: 10,170 rows | 1,017 participants | 288 sessions
env_natural_process_ctrl missing: 80 rows across 8 participants (climate-denial responses; excluded from the H4 climate-controls model only)


In [3]:
# Cell 3: Shared helpers for model estimation, extraction, and export
import patsy


def fit_lpm_mixedlm(df, formula, reml=True):
    """
    Three-level mixed-effects linear probability model:

    Level 1: decision rounds
    Level 2: participants
    Level 3: sessions

    Rows with missing values in formula variables are dropped 
    before estimation to ensure a consistent sample and avoid 
    grouping-related indexing errors.
    """
    _, design = patsy.dmatrices(formula, df, return_type='dataframe', NA_action='drop')
    fit_df = df.loc[design.index].dropna(subset=['sessioncode', 'participantcode'])
    model = MixedLM.from_formula(
        formula=formula,
        groups='sessioncode',
        re_formula='1',
        vc_formula={'participant': '0 + C(participantcode)'},
        data=fit_df
    )
    result = model.fit(reml=reml, method=['lbfgs', 'bfgs', 'cg', 'powell', 'nm'], maxiter=2000, disp=False)
    if not result.converged:
        print('WARNING: model did not converge with any optimizer in the retry sequence.')
    return result


def model_table(result, model_name, study_label):
    """
    Fixed-effect coefficients, standard errors, test statistics, p-values,
    and confidence intervals. Variance components are reported separately by
    variance_table.
    """
    k_fe = result.model.k_fe
    ci = result.conf_int()
    return pd.DataFrame({
        'term': result.params.index[:k_fe],
        'coef': result.params.values[:k_fe],
        'se': result.bse.values[:k_fe],
        'z': result.tvalues.values[:k_fe],
        'p': result.pvalues.values[:k_fe],
        'ci_low': ci[0].values[:k_fe],
        'ci_high': ci[1].values[:k_fe],
        'model': model_name,
        'study': study_label,
    })


def variance_table(result, model_name, study_label):
    rows = []
    if hasattr(result, 'cov_re') and result.cov_re is not None:
        try:
            rows.append({'component': 'session_intercept_var', 'value': float(np.asarray(result.cov_re)[0, 0])})
        except Exception:
            pass
    if hasattr(result, 'vcomp') and result.vcomp is not None:
        vals = np.atleast_1d(result.vcomp)
        for i, v in enumerate(vals, start=1):
            label = 'participant_intercept_var' if i == 1 else f'vc_formula_component_{i}'
            rows.append({'component': label, 'value': float(v)})
    rows.append({'component': 'residual_scale', 'value': float(result.scale)})
    out = pd.DataFrame(rows)
    out['model'] = model_name
    out['study'] = study_label
    return out


def fit_and_export(df, formula, model_name, study_label, prefix):
    result = fit_lpm_mixedlm(df, formula)
    coef_tbl = model_table(result, model_name, study_label)
    var_tbl = variance_table(result, model_name, study_label)
    slug = study_label.lower().replace(' ', '_').replace('(', '').replace(')', '')
    coef_tbl.to_csv(f'outputs/tables/{prefix}_{slug}_coefficients.csv', index=False)
    var_tbl.to_csv(f'outputs/tables/{prefix}_{slug}_variance_components.csv', index=False)
    with open(f'outputs/models/{prefix}_{slug}_summary.txt', 'w') as f:
        f.write(result.summary().as_text())
    return result, coef_tbl, var_tbl


## 3 · Moderation Specifications

Treatment enters directly as the four-level factorial assignment, interacted with each hypothesis's focal moderator:

`extract_green_choice ~ C(treatment, Treatment(reference=4)) * moderator + round_c`

The focal moderation terms are the interaction coefficients between each treatment contrast and the moderator. The `T.2` (IT-only) interaction is the cleanest test of each hypothesis, since it isolates intergenerational awareness from the green-information/monetary framing present in `IT&GI`.

Each hypothesis is estimated in two forms, following the same logic established for H1: a **primary** specification (lean, confirmatory) and a **robustness check** adding the full demographic covariate set, since none of the three moderators are randomly assigned.


In [4]:
# Cell 3: Model formulas for H2–H4 (primary and robustness-check specifications)
H2_FORMULA_PRIMARY = 'extract_green_choice ~ C(treatment, Treatment(reference=4)) * liberal_z + round_c'
H2_FORMULA_ROBUST_MT = (
    'extract_green_choice ~ C(treatment, Treatment(reference=4)) * liberal_z + round_c + '
    'age + C(gender) + C(education) + C(ethnic_min) + trust_scientists_z'
)
H2_FORMULA_ROBUST_PR = (
    'extract_green_choice ~ C(treatment, Treatment(reference=4)) * liberal_z + round_c + '
    'age + C(gender) + C(education) + C(ethnic_min) + trust_scientists_z + '
    'risk_attitude_z + short_long_reward_z'
)

H3_FORMULA_PRIMARY = 'extract_green_choice ~ C(treatment, Treatment(reference=4)) * legacy_z + round_c'
H3_FORMULA_ROBUST_PR = (
    'extract_green_choice ~ C(treatment, Treatment(reference=4)) * legacy_z + round_c + '
    'age + C(gender) + C(education) + C(ethnic_min) + trust_scientists_z + '
    'risk_attitude_z + short_long_reward_z'
)

H4_FORMULA_PRIMARY = 'extract_green_choice ~ C(treatment, Treatment(reference=4)) * env_concern_z + round_c'
H4_FORMULA_ROBUST_PR = (
    'extract_green_choice ~ C(treatment, Treatment(reference=4)) * env_concern_z + round_c + '
    'age + C(gender) + C(education) + C(ethnic_min) + trust_scientists_z + '
    'risk_attitude_z + short_long_reward_z'
)
# H4 climate-belief robustness: primary H4 augmented with the four single-item
# climate measures, isolating the env_concern_z moderation from correlated
# climate cognition.
H4_FORMULA_CLIMATE_PR = (
    'extract_green_choice ~ C(treatment, Treatment(reference=4)) * env_concern_z + round_c + '
    'environment_worry_z + environment_responsibility_z + '
    'C(environment_changing) + env_natural_process_ctrl_z'
)

print('H2 primary formula (both studies):')
print(H2_FORMULA_PRIMARY)
print('\nH2 full-controls robustness formula (MTurk):')
print(H2_FORMULA_ROBUST_MT)
print('\nH2 full-controls robustness formula (Prolific):')
print(H2_FORMULA_ROBUST_PR)
print('\nH3 primary formula (Prolific):')
print(H3_FORMULA_PRIMARY)
print('\nH3 full-controls robustness formula (Prolific):')
print(H3_FORMULA_ROBUST_PR)
print('\nH4 primary formula (Prolific):')
print(H4_FORMULA_PRIMARY)
print('\nH4 full-controls robustness formula (Prolific):')
print(H4_FORMULA_ROBUST_PR)
print('\nH4 climate-belief robustness formula (Prolific):')
print(H4_FORMULA_CLIMATE_PR)

H2 primary formula (both studies):
extract_green_choice ~ C(treatment, Treatment(reference=4)) * liberal_z + round_c

H2 full-controls robustness formula (MTurk):
extract_green_choice ~ C(treatment, Treatment(reference=4)) * liberal_z + round_c + age + C(gender) + C(education) + C(ethnic_min) + trust_scientists_z

H2 full-controls robustness formula (Prolific):
extract_green_choice ~ C(treatment, Treatment(reference=4)) * liberal_z + round_c + age + C(gender) + C(education) + C(ethnic_min) + trust_scientists_z + risk_attitude_z + short_long_reward_z

H3 primary formula (Prolific):
extract_green_choice ~ C(treatment, Treatment(reference=4)) * legacy_z + round_c

H3 full-controls robustness formula (Prolific):
extract_green_choice ~ C(treatment, Treatment(reference=4)) * legacy_z + round_c + age + C(gender) + C(education) + C(ethnic_min) + trust_scientists_z + risk_attitude_z + short_long_reward_z

H4 primary formula (Prolific):
extract_green_choice ~ C(treatment, Treatment(reference=4))

## 4 · H2 — Political Ideology as Moderator

Both studies. Primary specification first, then the robustness check with full covariate adjustment.


In [5]:
# Cell 4: H2 estimation in both studies (primary and robustness-check)
res_h2_mt, coef_h2_mt, var_h2_mt = fit_and_export(
    mt, H2_FORMULA_PRIMARY, model_name='H2 primary', study_label='Study 1 (MTurk)', prefix='h2'
)
res_h2_pr, coef_h2_pr, var_h2_pr = fit_and_export(
    pr, H2_FORMULA_PRIMARY, model_name='H2 primary', study_label='Study 2 (Prolific)', prefix='h2'
)

print('=== Study 1 (MTurk) - H2 Primary ===')
print(res_h2_mt.summary())
print()
print('=== Study 2 (Prolific) - H2 Primary ===')
print(res_h2_pr.summary())

res_h2_mt_robust, coef_h2_mt_robust, var_h2_mt_robust = fit_and_export(
    mt, H2_FORMULA_ROBUST_MT, model_name='H2 robustness (full controls)', study_label='Study 1 (MTurk)', prefix='h2_robust'
)
res_h2_pr_robust, coef_h2_pr_robust, var_h2_pr_robust = fit_and_export(
    pr, H2_FORMULA_ROBUST_PR, model_name='H2 robustness (full controls)', study_label='Study 2 (Prolific)', prefix='h2_robust'
)

print()
print('=== Study 1 (MTurk) - H2 Robustness check ===')
print(res_h2_mt_robust.summary())
print()
print('=== Study 2 (Prolific) - H2 Robustness check ===')
print(res_h2_pr_robust.summary())


=== Study 1 (MTurk) - H2 Primary ===
                            Mixed Linear Model Regression Results
Model:                      MixedLM          Dependent Variable:          extract_green_choice
No. Observations:           10220            Method:                      REML                
No. Groups:                 308              Scale:                       0.1461              
Min. group size:            10               Log-Likelihood:              -5386.1416          
Max. group size:            40               Converged:                   Yes                 
Mean group size:            33.2                                                              
----------------------------------------------------------------------------------------------
                                                    Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------------------------------------------
Intercept                                 

## 5 · H3 — Legacy Concern as Moderator (Prolific)

Legacy concern was collected in Prolific only. Primary specification first, then the robustness check.


In [6]:
# Cell 5: H3 estimation in Prolific only (primary and robustness-check)
res_h3_pr, coef_h3_pr, var_h3_pr = fit_and_export(
    pr, H3_FORMULA_PRIMARY, model_name='H3 primary', study_label='Study 2 (Prolific)', prefix='h3'
)
print('=== Study 2 (Prolific) - H3 Primary ===')
print(res_h3_pr.summary())

res_h3_pr_robust, coef_h3_pr_robust, var_h3_pr_robust = fit_and_export(
    pr, H3_FORMULA_ROBUST_PR, model_name='H3 robustness (full controls)', study_label='Study 2 (Prolific)', prefix='h3_robust'
)
print()
print('=== Study 2 (Prolific) - H3 Robustness check ===')
print(res_h3_pr_robust.summary())


=== Study 2 (Prolific) - H3 Primary ===
                            Mixed Linear Model Regression Results
Model:                     MixedLM          Dependent Variable:          extract_green_choice
No. Observations:          10170            Method:                      REML                
No. Groups:                288              Scale:                       0.1561              
Min. group size:           10               Log-Likelihood:              -5718.3113          
Max. group size:           40               Converged:                   Yes                 
Mean group size:           35.3                                                              
---------------------------------------------------------------------------------------------
                                                   Coef.  Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------------------------------------
Intercept                                       

## 6 · H4 — Environmental Concern as Moderator (Prolific)

Environmental concern was collected in Prolific only. Three specifications are estimated: the primary model, the full-demographic-controls robustness check, and a climate-belief robustness check augmenting the primary model with the four single-item climate measures. The climate-belief check listwise-drops the small number of climate-denial responses coded missing in preparation, so its analytic N is marginally lower than the other two.


In [7]:
# Cell 6: H4 estimation in Prolific only (primary, full-controls, and climate-belief robustness)
res_h4_pr, coef_h4_pr, var_h4_pr = fit_and_export(
    pr, H4_FORMULA_PRIMARY, model_name='H4 primary', study_label='Study 2 (Prolific)', prefix='h4'
)
print('=== Study 2 (Prolific) - H4 Primary ===')
print(res_h4_pr.summary())

res_h4_pr_robust, coef_h4_pr_robust, var_h4_pr_robust = fit_and_export(
    pr, H4_FORMULA_ROBUST_PR, model_name='H4 robustness (full controls)', study_label='Study 2 (Prolific)', prefix='h4_robust'
)
print()
print('=== Study 2 (Prolific) - H4 Robustness check (full demographic controls) ===')
print(res_h4_pr_robust.summary())

res_h4_pr_climate, coef_h4_pr_climate, var_h4_pr_climate = fit_and_export(
    pr, H4_FORMULA_CLIMATE_PR, model_name='H4 robustness (climate beliefs)', study_label='Study 2 (Prolific)', prefix='h4_climate'
)
print()
print('=== Study 2 (Prolific) - H4 Robustness check (climate-belief controls) ===')
print(res_h4_pr_climate.summary())


=== Study 2 (Prolific) - H4 Primary ===
                              Mixed Linear Model Regression Results
Model:                        MixedLM           Dependent Variable:           extract_green_choice
No. Observations:             10170             Method:                       REML                
No. Groups:                   288               Scale:                        0.1561              
Min. group size:              10                Log-Likelihood:               -5717.6434          
Max. group size:              40                Converged:                    Yes                 
Mean group size:              35.3                                                                
--------------------------------------------------------------------------------------------------
                                                        Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------------------------------------------------
I

## 7 · Focal Interaction Terms and Sensitivity Comparison

For each hypothesis, the focal inferential targets are the interaction coefficients between the moderator and the treatment contrasts; `T.2` (IT only) is the cleanest test of intergenerational awareness in isolation. The comparison table places primary and robustness-check estimates for these terms side by side, so any sensitivity to the added demographic covariates is visible directly.


In [8]:
# Cell 7: Extract focal moderation terms and build sensitivity comparison
FOCAL_MAP = {
    'H2': ['liberal_z',
           'C(treatment, Treatment(reference=4))[T.1]:liberal_z',
           'C(treatment, Treatment(reference=4))[T.2]:liberal_z',
           'C(treatment, Treatment(reference=4))[T.3]:liberal_z'],
    'H3': ['legacy_z',
           'C(treatment, Treatment(reference=4))[T.1]:legacy_z',
           'C(treatment, Treatment(reference=4))[T.2]:legacy_z',
           'C(treatment, Treatment(reference=4))[T.3]:legacy_z'],
    'H4': ['env_concern_z',
           'C(treatment, Treatment(reference=4))[T.1]:env_concern_z',
           'C(treatment, Treatment(reference=4))[T.2]:env_concern_z',
           'C(treatment, Treatment(reference=4))[T.3]:env_concern_z'],
}

for label, tbl, key in [
    ('H2 — Study 1 (MTurk), Primary', coef_h2_mt, 'H2'),
    ('H2 — Study 2 (Prolific), Primary', coef_h2_pr, 'H2'),
    ('H3 — Study 2 (Prolific), Primary', coef_h3_pr, 'H3'),
    ('H4 — Study 2 (Prolific), Primary', coef_h4_pr, 'H4'),
]:
    print(f'\n{label}')
    print(tbl[tbl['term'].isin(FOCAL_MAP[key])][['term', 'coef', 'se', 'z', 'p', 'ci_low', 'ci_high']].to_string(index=False))

# Primary vs. robustness-check comparison, focal terms only. H4 carries a
# third row set (climate-belief controls) in addition to the full-controls check.
comparison_rows = []
for hyp_key, spec_label, tbl in [
    ('H2', 'Primary', coef_h2_mt), ('H2', 'Robustness (full controls)', coef_h2_mt_robust),
]:
    sub = tbl[tbl['term'].isin(FOCAL_MAP[hyp_key])].copy()
    sub.insert(0, 'hypothesis', hyp_key)
    sub.insert(1, 'spec', spec_label)
    comparison_rows.append(sub)
for hyp_key, spec_label, tbl in [
    ('H2', 'Primary', coef_h2_pr), ('H2', 'Robustness (full controls)', coef_h2_pr_robust),
    ('H3', 'Primary', coef_h3_pr), ('H3', 'Robustness (full controls)', coef_h3_pr_robust),
    ('H4', 'Primary', coef_h4_pr), ('H4', 'Robustness (full controls)', coef_h4_pr_robust),
    ('H4', 'Robustness (climate beliefs)', coef_h4_pr_climate),
]:
    sub = tbl[tbl['term'].isin(FOCAL_MAP[hyp_key])].copy()
    sub.insert(0, 'hypothesis', hyp_key)
    sub.insert(1, 'spec', spec_label)
    comparison_rows.append(sub)

h2h4_comparison = pd.concat(comparison_rows, ignore_index=True)
h2h4_comparison = h2h4_comparison[['study', 'hypothesis', 'spec', 'term', 'coef', 'se', 'z', 'p', 'ci_low', 'ci_high']]
h2h4_comparison.to_csv('outputs/tables/h2_h4_primary_vs_robustness_comparison.csv', index=False)

print('\n\nH2-H4 focal terms: primary vs. robustness-check specifications')
print(h2h4_comparison.to_string(index=False))



H2 — Study 1 (MTurk), Primary
                                               term    coef     se       z      p  ci_low  ci_high
                                          liberal_z  0.0031 0.0139  0.2241 0.8227 -0.0240   0.0303
C(treatment, Treatment(reference=4))[T.1]:liberal_z -0.0232 0.0195 -1.1864 0.2355 -0.0614   0.0151
C(treatment, Treatment(reference=4))[T.2]:liberal_z  0.0011 0.0203  0.0519 0.9586 -0.0387   0.0408
C(treatment, Treatment(reference=4))[T.3]:liberal_z -0.0092 0.0199 -0.4627 0.6436 -0.0483   0.0298

H2 — Study 2 (Prolific), Primary
                                               term    coef     se       z      p  ci_low  ci_high
                                          liberal_z  0.0005 0.0161  0.0334 0.9734 -0.0311   0.0322
C(treatment, Treatment(reference=4))[T.1]:liberal_z -0.0093 0.0219 -0.4260 0.6701 -0.0522   0.0336
C(treatment, Treatment(reference=4))[T.2]:liberal_z -0.0208 0.0231 -0.9007 0.3678 -0.0660   0.0244
C(treatment, Treatment(reference=4))[T.3]:li

## 8 · Consolidated Outputs


In [9]:
# Cell 8: Consolidate all H2–H4 coefficient and variance-component outputs
coef_all = pd.concat([
    coef_h2_mt, coef_h2_pr, coef_h2_mt_robust, coef_h2_pr_robust,
    coef_h3_pr, coef_h3_pr_robust,
    coef_h4_pr, coef_h4_pr_robust, coef_h4_pr_climate,
], ignore_index=True)
var_all = pd.concat([
    var_h2_mt, var_h2_pr, var_h2_mt_robust, var_h2_pr_robust,
    var_h3_pr, var_h3_pr_robust,
    var_h4_pr, var_h4_pr_robust, var_h4_pr_climate,
], ignore_index=True)

coef_all.to_csv('outputs/tables/h2_h4_all_coefficients.csv', index=False)
var_all.to_csv('outputs/tables/h2_h4_all_variance_components.csv', index=False)

print('Exported:')
print(' outputs/tables/h2_h4_all_coefficients.csv')
print(' outputs/tables/h2_h4_all_variance_components.csv')
print(' outputs/tables/h2_h4_primary_vs_robustness_comparison.csv')
print(' outputs/models/h2*, h3*, h4*, h4_climate* summary files')


Exported:
 outputs/tables/h2_h4_all_coefficients.csv
 outputs/tables/h2_h4_all_variance_components.csv
 outputs/tables/h2_h4_primary_vs_robustness_comparison.csv
 outputs/models/h2*, h3*, h4*, h4_climate* summary files
